# Challenge — Make the Robot Write a Word

## The goal

Build something that takes a request like:

> **"write MHP"**

and makes the physical robot arm place each letter, in order, on its own — without you manually running a command per letter.

That's the whole brief. Everything below is context and constraints, not a recipe.

## What you're given

The **same two things** from Part 4 of the main workshop notebook — nothing more:

**The tunnel:**

```bash
ssh -N -L <PORT>:localhost:<PORT> ubuntu@<SERVER_IP>
```
Leave this running in its own terminal for the whole workshop — it's your private pipe into the shared GPU. `<PORT>` is whichever port your table was assigned (see above).

**The single-task command** (this is the only way you can move the robot — one call, one task):

```bash
python -m lerobot.async_inference.robot_client \
  --robot.type=so101_follower \
  --robot.port=<YOUR_ROBOT_PORT> \
  --robot.id=<YOUR_CALIBRATION_ID> \
  --robot.cameras="{ camera1: {type: opencv, index_or_path: <YOUR_CAMERA_INDEX>, width: 640, height: 480, fps: 30}}" \
  --server_address=127.0.0.1:<PORT> \
  --policy_type=smolvla \
  --pretrained_name_or_path=sohrabark/smolvla_abc_mhp_v2_merged_20260805 \
  --policy_device=cuda \
  --client_device=cpu \
  --actions_per_chunk=50 \
  --chunk_size_threshold=0.5 \
  --aggregate_fn_name=weighted_average \
  --fps=30 \
  --task="Pick up the M letter and place it in the first box."
```

Notes:
- `--robot.port`, `--robot.id`, `--robot.cameras` are the same values LeLab used for your calibration in Part 1.
- `--server_address` points at your **local** end of the SSH tunnel — `127.0.0.1`, not `<SERVER_IP>` — the tunnel does the forwarding.
- This command **runs until you press Ctrl-C** — it has no built-in stopping point. Let it run for ~20-25 seconds (that's roughly how long the training recordings were), then stop it.
- Swap `--task` for any of the three exact strings from the table above to try a different letter.

**The tasks it knows:**

| Letter | Task string (exact) | Goes into |
|---|---|---|
| **M** | `Pick up the M letter and place it in the first box.` | 1st box |
| **H** | `Pick up the H letter and place it in the second box.` | 2nd box |
| **P** | `Pick up the P letter and place it in the third box.` | 3rd box |

Only these three letters were trained. Anything else is not something the policy has ever seen.

## Constraints worth knowing before you plan

These are facts about the system, not hints about how to solve it:

- **Only M, H and P are trained.** If the word contains any other letter, the robot has never seen it — decide yourselves what "handling" that means (skip it? refuse the whole word? something else?).
- **The robot is one physical arm.** It can only run one task at a time — there's no such thing as placing two letters "at once".
- **The inference command has no built-in stop.** It runs until interrupted. You decide how long a letter's turn lasts, and how your program knows to move on to the next one.
- **You don't have to use an LLM.** A request like "write MHP" can be turned into `["M", "H", "P"]` with a single line of plain code — an LLM is one valid way to do that decomposition, not the only way. If you do use one, bring your own API key/credentials; none are provided.
- **You get to choose the language and tools.** Python, a shell script, an agent framework, whatever you're comfortable with.
- The policy needs a little time to "warm up" the first time it's asked to do something in a while.


Good luck — and watch the arm the whole time.